In [1]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils import check_X_y, check_array
from sklearn.datasets import make_classification, fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer


class GaussianNaiveBayes(BaseEstimator, ClassifierMixin):
    """
    Gaussian Naive Bayes classifier for continuous features.
    Assumes each feature follows a Gaussian distribution per class.
    """
    def __init__(self):
        self.classes_ = None
        self.class_priors_ = None        # P(y)
        self.means_ = None               # mean per class per feature
        self.variances_ = None           # variance per class per feature
        self.epsilon = 1e-9              # smoothing for variance

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        self.class_priors_ = np.zeros(n_classes)
        self.means_ = np.zeros((n_classes, n_features))
        self.variances_ = np.zeros((n_classes, n_features))

        for i, cls in enumerate(self.classes_):
            X_cls = X[y == cls]
            self.class_priors_[i] = X_cls.shape[0] / X.shape[0]
            self.means_[i] = np.mean(X_cls, axis=0)
            self.variances_[i] = np.var(X_cls, axis=0) + self.epsilon

        return self

    def _pdf(self, x, mean, var):
        """Gaussian probability density function."""
        coeff = 1.0 / np.sqrt(2.0 * np.pi * var)
        exponent = np.exp(-0.5 * ((x - mean) ** 2) / var)
        return coeff * exponent

    def predict_proba(self, X):
        X = check_array(X)
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        probas = np.zeros((n_samples, n_classes))

        for i, cls in enumerate(self.classes_):
            # Compute log of prior + sum of log likelihoods (use log to avoid underflow)
            log_prior = np.log(self.class_priors_[i])
            # For each sample, compute sum of log pdf over features
            log_likelihood = np.sum(
                np.log(self._pdf(X, self.means_[i], self.variances_[i]) + 1e-12),
                axis=1
            )
            log_posterior = log_prior + log_likelihood
            probas[:, i] = log_posterior

        # Convert log probabilities to actual probabilities using softmax
        # Subtract max for numerical stability
        max_log = np.max(probas, axis=1, keepdims=True)
        exp_logs = np.exp(probas - max_log)
        probas = exp_logs / np.sum(exp_logs, axis=1, keepdims=True)
        return probas

    def predict(self, X):
        probas = self.predict_proba(X)
        return self.classes_[np.argmax(probas, axis=1)]


class MultinomialNaiveBayes(BaseEstimator, ClassifierMixin):
    """
    Multinomial Naive Bayes for discrete count features (e.g., bag-of-words).
    Uses Laplace smoothing.
    """
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_priors_ = None
        self.feature_log_probs_ = None   # log P(feature | class)

    def fit(self, X, y):
        X, y = check_X_y(X, y, accept_sparse=False)  # we expect dense array
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        self.class_priors_ = np.zeros(n_classes)
        self.feature_log_probs_ = np.zeros((n_classes, n_features))

        for i, cls in enumerate(self.classes_):
            X_cls = X[y == cls]
            self.class_priors_[i] = X_cls.shape[0] / X.shape[0]
            # Sum of feature counts for this class
            feature_counts = X_cls.sum(axis=0)
            total_count = feature_counts.sum()
            # Laplace smoothing: (count + alpha) / (total_count + alpha * n_features)
            smoothed_probs = (feature_counts + self.alpha) / (total_count + self.alpha * n_features)
            self.feature_log_probs_[i] = np.log(smoothed_probs)

        return self

    def predict_proba(self, X):
        X = check_array(X)
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        log_probas = np.zeros((n_samples, n_classes))

        for i, cls in enumerate(self.classes_):
            log_prior = np.log(self.class_priors_[i])
            # log likelihood: sum over features of x_j * log P(feature_j | class)
            log_likelihood = X @ self.feature_log_probs_[i].T  # dot product
            log_probas[:, i] = log_prior + log_likelihood

        # Softmax to get probabilities
        max_log = np.max(log_probas, axis=1, keepdims=True)
        exp_logs = np.exp(log_probas - max_log)
        probas = exp_logs / np.sum(exp_logs, axis=1, keepdims=True)
        return probas

    def predict(self, X):
        probas = self.predict_proba(X)
        return self.classes_[np.argmax(probas, axis=1)]



# Example usage

if __name__ == "__main__":
    # ----- 1. Gaussian Naive Bayes on synthetic data -----
    print("--- Gaussian Naive Bayes ---")
    X, y = make_classification(n_samples=500, n_features=5, n_informative=4,
                               n_redundant=1, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    gnb = GaussianNaiveBayes()
    gnb.fit(X_train, y_train)
    y_pred = gnb.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy on synthetic data: {acc:.3f}")

    # ----- 2. Multinomial Naive Bayes on text data (20 Newsgroups) -----
    print("\n--- Multinomial Naive Bayes (text classification) ---")
    # Load a small subset of 20 newsgroups for quick demo
    categories = ['alt.atheism', 'soc.religion.christian']
    newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                    shuffle=True, random_state=42)
    X_text = newsgroups.data
    y_text = newsgroups.target

    # Vectorize: convert text to bag-of-words counts
    vectorizer = CountVectorizer(stop_words='english', max_features=1000)
    X_counts = vectorizer.fit_transform(X_text).toarray()  # dense for simplicity

    X_train, X_test, y_train, y_test = train_test_split(
        X_counts, y_text, test_size=0.3, random_state=42
    )

    mnb = MultinomialNaiveBayes(alpha=1.0)
    mnb.fit(X_train, y_train)
    y_pred = mnb.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy on 20 Newsgroups (2 classes): {acc:.3f}")

    # Show predicted probabilities for a few test samples
    probas = mnb.predict_proba(X_test[:3])
    print("Predicted probabilities for first 3 test samples:\n", probas)

--- Gaussian Naive Bayes ---
Accuracy on synthetic data: 0.720

--- Multinomial Naive Bayes (text classification) ---
Accuracy on 20 Newsgroups (2 classes): 0.944
Predicted probabilities for first 3 test samples:
 [[8.88076963e-37 1.00000000e+00]
 [1.63618657e-31 1.00000000e+00]
 [1.19807742e-18 1.00000000e+00]]
